# Arithmetic Coding

Arithmetic coding (Rissanen, 1976) is a lossless compression technique that encodes an entire message as a **single fractional number** in the interval [0, 1). Unlike Huffman coding, which assigns a fixed code to each symbol, arithmetic coding represents the whole message by successively **subdividing an interval** based on symbol probabilities.

Why it's interesting:

- It achieves compression **arbitrarily close to the Shannon entropy** — the theoretical optimum — without requiring symbol probabilities to be powers of 1/2.
- The core idea is elegant: each symbol narrows the current interval proportionally to its probability. The final interval uniquely identifies the message.
- Key concepts: **cumulative probability distribution** (determines where each symbol's sub-interval starts) and **interval subdivision** (each new symbol refines [low, high) into a smaller sub-interval).

## Implementation

We use Python's `fractions.Fraction` for exact rational arithmetic — no rounding errors, so the algorithm is transparent and easy to follow. A simple static probability model maps each symbol to its frequency. Encoding narrows an interval; decoding inverts the process.

In [1]:
from fractions import Fraction
from collections import Counter
import math


class ArithmeticModel:
    """A static probability model built from symbol frequencies."""

    def __init__(self, frequencies):
        total = sum(frequencies.values())
        self.prob = {}
        self.cumulative = {}
        cum = Fraction(0)
        for symbol in sorted(frequencies):
            self.cumulative[symbol] = cum
            self.prob[symbol] = Fraction(frequencies[symbol], total)
            cum += self.prob[symbol]
        self.symbols = sorted(frequencies)

    @classmethod
    def from_string(cls, text):
        return cls(Counter(text))


def arithmetic_encode(message, model):
    """Encode a message into an interval [low, high) within [0, 1)."""
    low = Fraction(0)
    high = Fraction(1)
    for ch in message:
        width = high - low
        high = low + width * (model.cumulative[ch] + model.prob[ch])
        low = low + width * model.cumulative[ch]
    return low, high


def arithmetic_decode(encoded, length, model):
    """Decode a fractional value back into a message of the given length."""
    result = []
    for _ in range(length):
        for symbol in model.symbols:
            sym_low = model.cumulative[symbol]
            sym_high = sym_low + model.prob[symbol]
            if sym_low <= encoded < sym_high:
                result.append(symbol)
                # Zoom into this symbol's sub-interval
                encoded = (encoded - sym_low) / (sym_high - sym_low)
                break
    return ''.join(result)

## Example: Step by Step

Let's encode the string `"ABBA"` with equal symbol probabilities and watch the interval narrow.

In [2]:
message = "ABBA"
model = ArithmeticModel({"A": 2, "B": 2})

print(f"Model: A -> [{model.cumulative['A']}, {model.cumulative['A'] + model.prob['A']})"
      f"  B -> [{model.cumulative['B']}, {model.cumulative['B'] + model.prob['B']})")

low = Fraction(0)
high = Fraction(1)
print(f"\nStart:       [{low}, {high})")

for ch in message:
    width = high - low
    high = low + width * (model.cumulative[ch] + model.prob[ch])
    low = low + width * model.cumulative[ch]
    print(f"After '{ch}':   [{low}, {high}){' ' * (20 - len(str(high)))}width = {high - low}")

midpoint = (low + high) / 2
print(f"\nAny value in [{low}, {high}) encodes \"{message}\".")
print(f"Midpoint: {midpoint} = {float(midpoint)}")

Model: A -> [0, 1/2)  B -> [1/2, 1)

Start:       [0, 1)
After 'A':   [0, 1/2)                 width = 1/2
After 'B':   [1/4, 1/2)                 width = 1/4
After 'B':   [3/8, 1/2)                 width = 1/8
After 'A':   [3/8, 7/16)                width = 1/16

Any value in [3/8, 7/16) encodes "ABBA".
Midpoint: 13/32 = 0.40625


In [3]:
# Verify encode/decode round-trip.
low, high = arithmetic_encode(message, model)
encoded_value = (low + high) / 2
decoded = arithmetic_decode(encoded_value, len(message), model)

assert decoded == message, f"Expected {message!r}, got {decoded!r}"
print(f"Encoded: {message} -> [{low}, {high})")
print(f"Decoded: {encoded_value} -> {decoded}")
print("Round-trip OK.")

Encoded: ABBA -> [3/8, 7/16)
Decoded: 13/32 -> ABBA
Round-trip OK.


## Longer Example

With a longer, non-uniform string we can measure actual compression and compare to the Shannon entropy bound.

In [4]:
text = "hello world"
model2 = ArithmeticModel.from_string(text)

print(f'Message: "{text}" ({len(text)} symbols)\n')
print("Symbol frequencies:")
for s in model2.symbols:
    print(f"  {s!r:3s} : {model2.prob[s]}")

low, high = arithmetic_encode(text, model2)
interval_size = high - low
bits_needed = math.ceil(-math.log2(float(interval_size)))
original_bits = len(text) * 8

freq = Counter(text)
total = len(text)
entropy = -sum((c / total) * math.log2(c / total) for c in freq.values())
theoretical_min = math.ceil(entropy * total)

print(f"\nOriginal (ASCII)       : {original_bits} bits")
print(f"Encoded (arithmetic)   : {bits_needed} bits")
print(f"Shannon entropy        : {entropy:.4f} bits/symbol")
print(f"Theoretical minimum    : {theoretical_min} bits")
print(f"Compression ratio      : {bits_needed / original_bits:.2f}")

encoded_value = (low + high) / 2
decoded = arithmetic_decode(encoded_value, len(text), model2)
assert decoded == text
print(f"\nRound-trip OK.")

Message: "hello world" (11 symbols)

Symbol frequencies:
  ' ' : 1/11
  'd' : 1/11
  'e' : 1/11
  'h' : 1/11
  'l' : 3/11
  'o' : 2/11
  'r' : 1/11
  'w' : 1/11

Original (ASCII)       : 88 bits
Encoded (arithmetic)   : 32 bits
Shannon entropy        : 2.8454 bits/symbol
Theoretical minimum    : 32 bits
Compression ratio      : 0.36

Round-trip OK.


## How It Works — Recap

1. **Build a probability model.** Assign each symbol a sub-interval of [0, 1) proportional to its frequency. The intervals are contiguous and non-overlapping.
2. **Encode: subdivide the interval.** Start with [0, 1). For each symbol in the message, narrow [low, high) to the portion corresponding to that symbol's sub-interval. After all symbols, the final [low, high) uniquely identifies the message.
3. **Pick a representative value.** Any number in [low, high) suffices. In practice, choose one with a short binary representation (fewest bits).
4. **Decode: invert the process.** Given the encoded value and message length, repeatedly determine which symbol's sub-interval contains the current value, emit that symbol, and rescale into that sub-interval.
5. **Near-optimal compression.** The width of the final interval equals the product of all symbol probabilities — exactly 2^(-H) when the model matches the true distribution. The number of bits needed is ceil(-log2(width)), which approaches the Shannon entropy H.

## References

- Rissanen, J. (1976). **"Generalized Kraft Inequality and Arithmetic Coding."** *IBM Journal of Research and Development*, 20(3), 198-203. [https://doi.org/10.1147/rd.203.0198](https://doi.org/10.1147/rd.203.0198)
- Witten, I.H., Neal, R.M., and Cleary, J.G. (1987). **"Arithmetic Coding for Data Compression."** *Communications of the ACM*, 30(6), 520-540. [https://doi.org/10.1145/214762.214771](https://doi.org/10.1145/214762.214771)
- [Wikipedia: Arithmetic coding](https://en.wikipedia.org/wiki/Arithmetic_coding)